**Note:**  
This notebook expects the raw dataset to be located at `../data/fraud_data_raw.csv`.  
Please place the dataset in the `data1/` folder before running.

## 03 — Modeling for Fraud Detection

In this notebook, we build and evaluate machine learning models to detect fraudulent transactions.
We will:
- Load the cleaned dataset
- Split the data into training and testing sets
- Handle class imbalance
- Train baseline models
- Evaluate using fraud-appropriate metrics (precision, recall, F1, ROC-AUC)
- Compare model performance
- Select a final model for deployment

In [1]:
#Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

pd.set_option("display.max_columns", None)

In [3]:
#Loading Cleaned Dataset

df = pd.read_csv("../data/fraud_data_cleaned.csv")
df.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,balanceDiffOrg,balanceDiffDest
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,False,False,True,False,9839.64,0.0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,False,False,True,False,1864.28,0.0
2,1,181.00,181.0,0.00,0.0,0.0,1,False,False,False,True,181.00,0.0
3,1,181.00,181.0,0.00,21182.0,0.0,1,True,False,False,False,181.00,-21182.0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,False,False,True,False,11668.14,0.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 13 columns):
 #   Column           Dtype  
---  ------           -----  
 0   step             int64  
 1   amount           float64
 2   oldbalanceOrg    float64
 3   newbalanceOrig   float64
 4   oldbalanceDest   float64
 5   newbalanceDest   float64
 6   isFraud          int64  
 7   type_CASH_OUT    bool   
 8   type_DEBIT       bool   
 9   type_PAYMENT     bool   
 10  type_TRANSFER    bool   
 11  balanceDiffOrg   float64
 12  balanceDiffDest  float64
dtypes: bool(4), float64(7), int64(2)
memory usage: 461.2 MB


In [5]:
#Defining features and target
##"isFraud" is the target variable.
##All other columns are features.

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X.shape, y.shape

((6362620, 12), (6362620,))

In [6]:
##Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#Base Model 1: Logitstic Regression
# Fraud is rare, so we use class weights to give more weight to the fraud cases, to handle class imbalance.

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)

log_pred = log_reg.predict(X_test)
log_proba = log_reg.predict_proba(X_test)[:, 1]


In [9]:
#Evaluation: Logistic Regression

print("Classification Report:")
print(classification_report(y_test, log_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, log_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, log_proba))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98   1270881
           1       0.03      0.92      0.06      1643

    accuracy                           0.96   1272524
   macro avg       0.51      0.94      0.52   1272524
weighted avg       1.00      0.96      0.98   1272524


Confusion Matrix:
[[1221703   49178]
 [    128    1515]]

ROC-AUC Score:
0.9873744115214093


In [10]:
#Baseline Model 2: Random Forest

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]


In [11]:
#Evaluation: Random Forest

print("Classification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, rf_proba))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.91      0.88      0.90      1643

    accuracy                           1.00   1272524
   macro avg       0.96      0.94      0.95   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:
[[1270740     141]
 [    192    1451]]

ROC-AUC Score:
0.9987128907983228


In [12]:
#Model Comparison

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "ROC-AUC": [
        roc_auc_score(y_test, log_proba),
        roc_auc_score(y_test, rf_proba)
    ]
})

results

,Model,ROC-AUC
0,Logistic Regression,0.987374
1,Random Forest,0.998713


## Choosing the model with the best "fraud-focused" performance 

We chose the Random Forest model because it achieves both high precision (0.91) and high recall (0.88) for fraud cases, meaning it correctly identifies fraud while minimizing false alarms. Logistic Regression reaches high recall but extremely low precision (0.03), producing tens of thousands of false fraud alerts. Random Forest maintains a far lower false positive count (141 vs. 49,178), which is critical in fraud detection where investigation resources are limited. Its fraud-class F1 score (0.90) is substantially higher than Logistic Regression’s (0.06), showing balanced performance. Random Forest also achieves the highest ROC-AUC (0.9987), indicating stronger overall discrimination between fraud and non-fraud transactions.

